
# Dynamic AI Harm Map — Incremental Inference

Use this notebook when you have a JSON file containing **ONLY NEW REPORTS**.

## What happens

1. Mount Google Drive.
2. Load the existing persistent extraction file:
   `MyDrive/ai_harm_ground_truth_outputs/predicted_events.jsonl`
3. Upload/read the JSON containing only the new reports.
4. Compare `report_id`s.
5. Qwen extracts **only IDs not already present**.
6. Every successful new event is physically **APPENDED TO THE END** of the existing
   `predicted_events.jsonl`. Existing lines are never rewritten or reordered.
7. Reload the combined old + new extraction corpus.
8. Run the narrow harm-category **fit / not_fit / uncertain** audit.
9. Load the existing trained `predicted_phtkg.pt` — no graph retraining.
10. Run checkpoint inference, recurrence, incremental temporal evolution,
    fast latent-pattern clustering, similarity, novelty and pattern evolution.
11. Write one final enriched JSONL containing **old events first, then the newly appended events**.

### Important

- The uploaded source JSON may contain only new data.
- Re-running the notebook is safe: existing `report_id`s are skipped.
- The class verifier **cannot choose another taxonomy class**.
- The existing PHTKG checkpoint was trained before this new verification pass, so
  checkpoint inference keeps the original supplied `harm_category`.
- `verified_harm_category` is also saved. A separate verified dataset is produced
  for the final retrain if you want class filtering to affect the trained graph.


In [ ]:

# ============================================================
# 0. INSTALL
# ============================================================
!pip -q install -U "transformers>=4.45" accelerate sentencepiece json-repair openpyxl scikit-learn


In [ ]:

# ============================================================
# 1. DRIVE + GLOBAL CONFIG
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os, json, copy, shutil, re, math, hashlib, gc, random, time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ai_harm_ground_truth_outputs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This is your persistent OLD + NEW extraction history.
PREDICTED_EVENTS_PATH = DRIVE_OUTPUT_DIR / "predicted_events.jsonl"

# Existing trained checkpoint.
PHTKG_CHECKPOINT_PATH = DRIVE_OUTPUT_DIR / "predicted_phtkg.pt"

# Authoritative taxonomy.
TAXONOMY_PATH = DRIVE_OUTPUT_DIR / "AI_Harm_Map_Taxonomy_Schema_vSHARED (1).xlsx"
TAXONOMY_SHEET = "Taxonomy & Schema"

# Persistent incremental outputs.
INFERENCE_DIR = DRIVE_OUTPUT_DIR / "incremental_inference"
INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DIR = DRIVE_OUTPUT_DIR / "snapshots"
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- extraction speed / quality ----------------
QWEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EXTRACTION_BATCH_SIZE = 4
MAX_CONTEXT_TOKENS = 7000

# Fast first attempt; automatically retry only capped/malformed records at 750.
FAST_MAX_NEW_TOKENS = 500
RETRY_MAX_NEW_TOKENS = 750

SNAPSHOT_EVERY_NEW = 25

# ---------------- category-fit pass ----------------
RUN_CLASS_FIT = True
VERIFY_BATCH_SIZE = 16
VERIFY_MAX_INPUT_TOKENS = 1800
VERIFY_MAX_NEW_TOKENS = 120
VERIFY_AUDIT_PATH = INFERENCE_DIR / "harm_category_fit_audit.jsonl"
ACCEPTED_CLASS_STATUSES = {"fit"}

# ---------------- PHTKG / analysis ----------------
TOP_K_SIMILAR = 5
MIN_PATTERN_SIZE = 5
COARSE_K_POINTS = 18
KMEANS_N_INIT = 20
FREQUENCY_TREND_THRESHOLD = 0.15

FINAL_ENRICHED_JSONL = (
    INFERENCE_DIR / "predicted_events_ALL_with_verified_class_and_phtkg.jsonl"
)
FINAL_ENRICHED_JSON = FINAL_ENRICHED_JSONL.with_suffix(".json")

print("Device:", DEVICE)
print("Persistent extraction history:", PREDICTED_EVENTS_PATH)
print("Incremental outputs:", INFERENCE_DIR)


In [ ]:

# ============================================================
# 2. UPLOAD / FIND THE JSON CONTAINING ONLY NEW REPORTS
# ============================================================
from google.colab import files

def find_new_input_json():
    candidates = []
    for pattern in [
        "new_data*.json",
        "new_combined_data*.json",
        "*new*data*.json",
        "*combined*.json",
    ]:
        candidates.extend(Path("/content").glob(pattern))

    blocked = {
        "predicted_events.jsonl",
        "predicted_events.json",
    }

    candidates = [
        p for p in candidates
        if p.is_file()
        and p.name not in blocked
        and "predicted_events" not in p.name.lower()
    ]

    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

NEW_INPUT_PATH = find_new_input_json()

if NEW_INPUT_PATH is None:
    print("Upload the JSON containing ONLY the new reports.")
    uploaded = files.upload()
    json_files = [
        Path("/content") / name
        for name in uploaded
        if name.lower().endswith(".json")
    ]
    if not json_files:
        raise FileNotFoundError("No JSON file was uploaded.")
    NEW_INPUT_PATH = max(json_files, key=lambda p: p.stat().st_size)

with NEW_INPUT_PATH.open("r", encoding="utf-8") as f:
    new_reports_source = json.load(f)

if not isinstance(new_reports_source, list):
    raise ValueError("New input JSON must contain a top-level list of reports.")

new_reports_source = [
    row for row in new_reports_source
    if isinstance(row, dict) and str(row.get("report_id", "")).strip()
]

new_ids = [str(row["report_id"]).strip() for row in new_reports_source]

if len(new_ids) != len(set(new_ids)):
    duplicates = [
        rid for rid, count in Counter(new_ids).items()
        if count > 1
    ]
    raise ValueError(f"Duplicate report_id values in NEW input: {duplicates[:20]}")

new_reports_by_id = {
    str(row["report_id"]).strip(): row
    for row in new_reports_source
}

print("New-source file:", NEW_INPUT_PATH)
print("Rows supplied in new JSON:", len(new_reports_source))
print("First new-source ID:", new_ids[0] if new_ids else None)
print("Last new-source ID:", new_ids[-1] if new_ids else None)


In [ ]:
# ============================================================
# 3. EXTRACTION PROMPT
# ============================================================

EXTRACTION_PROMPT = r"""
You are the event-extraction component of a multilingual AI-harm knowledge-graph pipeline.

TASK
----
The input report is already part of a curated AI-harm corpus. Extract exactly ONE central
AI-related harm event from the report. Read the entire original-language report and the supplied
English translation before deciding that a field is unavailable.

Return only the structured event fields requested below. Source evidence and taxonomy
classification are attached separately from the input record after extraction.

GROUNDING RULES
---------------
1. Use only information supported by the supplied report and metadata.
2. Never invent an organization, AI system, affected group, action, consequence, location, or date.
3. Preserve attribution and modality. If the source says alleged, claimed, may, could, planned,
   proposed, disputed, or denied, do not convert that statement into an established fact.
4. Do not combine organizations, locations, actions, or affected groups from different incidents
   or geographic examples in the same report.
5. Do not treat a regulator, court, complainant, journalist, advocacy group, investigation,
   lawsuit, consultation, remedy, ban, or corrective action as the harmful actor/action unless
   that entity or action itself caused the documented harm.
6. Identify the actor responsible for the underlying harmful AI-related conduct.
7. Keep ACTION and CONSEQUENCE distinct:
   - action = what the AI-enabled system or responsible actor did;
   - consequence = the adverse effect, risk, restriction, exposure, inequality, surveillance,
     misinformation effect, exploitation, safety problem, or other documented harm.
8. If a factual field genuinely cannot be established, use exactly:
   "Not specified in report"
9. Never return JSON null or an empty required field.
10. Return exactly one JSON object and no markdown or explanation.

FIELD RULES
-----------

event_type
    A short 2-8 word descriptive title for the concrete incident or harm. Do not use a generic
    label such as "ai_harm_event".

ai_system
    The AI system, model, algorithm, automated tool, or AI-enabled process actually involved.
    A source-supported generic descriptor such as "AI chatbot", "facial recognition technology",
    "automated hiring algorithm", "recommender system", or "AI system" is valid when no product
    name is given.

organization
    The organization, authority, company, institution, platform, government body, or other actor
    that deployed, operated, purchased, commissioned, or used the AI-enabled system. Do not use
    the article publisher merely because it is named.

affected_group
    The most specific source-supported person, population, institution, community, or group
    affected or placed at risk.

action
    A concise description of the harmful AI-enabled conduct or use. Do not output a later remedy
    or response when the report describes an earlier harmful action.

consequence
    The adverse outcome, risk, exposure, restriction, denial, inequality, privacy intrusion,
    surveillance, misinformation effect, exploitation, discrimination, safety problem, or other
    harm supported by the report. Do not simply repeat the action.

location
    The most specific source-supported geographic or jurisdictional context. Do not infer a
    location from an unrelated example in the report.

event_date
    Use the explicit event date when available, then month/year or year, then an explicit range.
    If the report describes a current/recent event without a separate occurrence date, use the
    supplied publication date as fallback. Use YYYY-MM-DD, YYYY-MM, or YYYY and do not invent
    missing components.

confidence
    A number from 0 to 1 reflecting how directly the event fields are supported by the report.

CLASSIFICATION
--------------
Do not output harm_category. The supplied classification is copied separately from the source
record and verified in a later class-fit stage.

OUTPUT FORMAT
-------------
Return exactly:

{
  "harm_event": {
    "event_type": "...",
    "ai_system": "...",
    "organization": "...",
    "affected_group": "...",
    "action": "...",
    "consequence": "...",
    "location": "...",
    "event_date": "...",
    "confidence": 0.0
  }
}

Return JSON only.
"""


In [ ]:

# ============================================================
# 4. APPEND-ONLY EXTRACTION HELPERS
# ============================================================
from json_repair import repair_json

def clean(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()

def normalize(value):
    return " ".join(re.sub(r"[^\w\s]", " ", clean(value).casefold()).split())

def _missing(value):
    return value is None or value == "" or value == []

def supplied_classification(report):
    # Copy upstream data exactly; Qwen never predicts it.
    raw = report.get("classifications")
    source_field = "classifications"

    if _missing(raw):
        raw = report.get("classification")
        source_field = "classification"

    if _missing(raw):
        return [], None

    return copy.deepcopy(raw), source_field

def parse_json_object(text):
    text = clean(text)
    if not text:
        raise ValueError("Empty model output.")

    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    candidate = text[start:end + 1] if start >= 0 and end > start else text

    try:
        obj = json.loads(candidate)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    repaired = repair_json(candidate, return_objects=True)
    if not isinstance(repaired, dict):
        raise ValueError("Could not recover a JSON object.")
    return repaired

def normalize_extracted_event(parsed, report, generated_tokens, generation_budget, batch_size):
    event = parsed.get("harm_event")
    if not isinstance(event, dict):
        raise ValueError("Model did not return harm_event object.")

    required = [
        "event_type",
        "ai_system",
        "organization",
        "affected_group",
        "action",
        "consequence",
        "location",
        "event_date",
    ]

    out = {}
    for field in required:
        value = event.get(field)
        out[field] = value if value is not None and clean(value) else "Not specified in report"

    try:
        confidence = float(event.get("confidence", 0.0))
        confidence = float(np.clip(confidence, 0.0, 1.0))
    except Exception:
        confidence = 0.0
    out["confidence"] = confidence

    original_text = clean(report.get("original_text"))
    translated_text = clean(report.get("translated_text"))
    source_language = clean(report.get("source_language")).lower()

    out["original_evidence_span"] = original_text or "Not specified in report"
    out["translated_evidence_span"] = (
        original_text if source_language == "en" and original_text
        else translated_text if source_language != "en" and translated_text
        else "Not specified in report"
    )
    out["original_text"] = original_text
    out["translated_text"] = original_text if source_language == "en" else translated_text

    publication_date = clean(report.get("publication_date"))
    out["event_date_type"] = (
        "publication_date_fallback"
        if publication_date and clean(out["event_date"]) == publication_date
        else "event_date"
    )

    out["report_id"] = clean(report.get("report_id"))
    out["source"] = clean(report.get("source_url")) or "Not specified in report"

    category, source_field = supplied_classification(report)
    out["harm_category"] = category
    out["taxonomy_audit"] = {
        "method": "direct_source_json_copy",
        "source_field": source_field,
        "model_selected_classification": False,
    }
    out["evidence_audit"] = {
        "method": "full_source_text_copy",
        "source_field": "original_text",
    }
    out["inference_audit"] = {
        "mode": "incremental_single_pass_batched",
        "generated_tokens": int(generated_tokens),
        "generation_budget": int(generation_budget),
        "batch_size": int(batch_size),
        "max_input_tokens": int(MAX_CONTEXT_TOKENS),
        "appended_incrementally": True,
    }

    return out

def read_jsonl_rows(path):
    rows = []
    if not Path(path).exists():
        return rows

    with Path(path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception as exc:
                print(f"WARNING: damaged line {line_no} skipped: {exc}")
                continue
            if clean(row.get("report_id")):
                rows.append(row)
    return rows

def unique_rows_preserve_first_position(rows):
    # Latest valid value wins, but first physical position is preserved.
    order = []
    by_id = {}
    for row in rows:
        rid = clean(row.get("report_id"))
        if not rid:
            continue
        if rid not in by_id:
            order.append(rid)
        by_id[rid] = row
    return [by_id[rid] for rid in order]

def append_event_durable(path, row):
    # Critical: this is APPEND ONLY. Existing rows are never rebuilt/reordered.
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with Path(path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass

def make_full_snapshot(path, expected_total_count=None):
    path = Path(path)
    if not path.exists():
        return None

    rows = read_jsonl_rows(path)
    unique_rows = unique_rows_preserve_first_position(rows)
    actual_count = len(unique_rows)

    if expected_total_count is not None and actual_count != expected_total_count:
        raise RuntimeError(
            f"Snapshot count mismatch: expected {expected_total_count}, found {actual_count}."
        )

    snapshot = SNAPSHOT_DIR / f"predicted_events_combined_{actual_count:05d}.jsonl"
    shutil.copy2(path, snapshot)
    return snapshot

existing_rows_before_run = read_jsonl_rows(PREDICTED_EVENTS_PATH)
existing_unique_before_run = unique_rows_preserve_first_position(existing_rows_before_run)
existing_ids_before_run = {
    clean(row["report_id"])
    for row in existing_unique_before_run
}

new_reports_to_extract = [
    row for row in new_reports_source
    if clean(row.get("report_id")) not in existing_ids_before_run
]

already_present_from_new_file = [
    clean(row.get("report_id"))
    for row in new_reports_source
    if clean(row.get("report_id")) in existing_ids_before_run
]

print("Already stored before this run:", len(existing_unique_before_run))
print("Rows supplied in NEW input:", len(new_reports_source))
print("Already present -> skipped:", len(already_present_from_new_file))
print("Actually new -> Qwen extraction:", len(new_reports_to_extract))

if already_present_from_new_file:
    print("First skipped IDs:", already_present_from_new_file[:10])


In [ ]:

# ============================================================
# 5. FAST QWEN EXTRACTOR
# Batch=4, SDPA, 7000 context.
# 500-token first attempt; capped/malformed rows retry alone at 750.
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

class IncrementalQwenExtractor:
    def __init__(self, model_name):
        print("Loading Qwen for new-event extraction...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        kwargs = {
            "device_map": "auto",
            "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
        }

        if torch.cuda.is_available():
            kwargs["attn_implementation"] = "sdpa"

        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
        except Exception as exc:
            if "attn_implementation" in kwargs:
                print("SDPA load failed; falling back to default attention:", type(exc).__name__)
                kwargs.pop("attn_implementation", None)
                self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
            else:
                raise

        self.model.eval()
        print("Qwen loaded.")

    def build_prompt(self, report):
        user_text = f'''
REPORT ID:
{clean(report.get("report_id"))}

SOURCE:
{clean(report.get("source_url"))}

PUBLICATION DATE:
{clean(report.get("publication_date"))}

SOURCE LANGUAGE:
{clean(report.get("source_language"))}

COUNTRY / DATASET LOCATION HINT:
{clean(report.get("country"))}

ORIGINAL-LANGUAGE REPORT:
{clean(report.get("original_text"))}

CANONICAL ENGLISH TRANSLATION:
{clean(report.get("translated_text"))}
'''.strip()

        messages = [
            {"role": "system", "content": EXTRACTION_PROMPT},
            {"role": "user", "content": user_text},
        ]
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def generate_batch(self, reports_batch, max_new_tokens):
        prompts = [self.build_prompt(r) for r in reports_batch]

        encoded = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_CONTEXT_TOKENS,
        )

        device = next(self.model.parameters()).device
        encoded = {k: v.to(device) for k, v in encoded.items()}
        input_width = encoded["input_ids"].shape[1]

        with torch.inference_mode():
            generated = self.model.generate(
                **encoded,
                max_new_tokens=int(max_new_tokens),
                do_sample=False,
                use_cache=True,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id,
            )

        outputs = []
        eos_id = self.tokenizer.eos_token_id

        for sequence in generated:
            new_ids = sequence[input_width:].detach().cpu().tolist()

            try:
                eos_pos = new_ids.index(eos_id)
                answer_ids = new_ids[:eos_pos]
                hit_ceiling = False
            except ValueError:
                answer_ids = new_ids
                hit_ceiling = len(answer_ids) >= int(max_new_tokens)

            outputs.append({
                "text": self.tokenizer.decode(answer_ids, skip_special_tokens=True),
                "generated_tokens": len(answer_ids),
                "hit_ceiling": bool(hit_ceiling),
            })

        del encoded, generated
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return outputs

    def generate_recursive(self, reports_batch, max_new_tokens):
        if not reports_batch:
            return []

        try:
            outputs = self.generate_batch(reports_batch, max_new_tokens)
            return list(zip(reports_batch, outputs))
        except torch.cuda.OutOfMemoryError:
            if len(reports_batch) == 1:
                raise
            torch.cuda.empty_cache()
            mid = len(reports_batch) // 2
            print(f"OOM: splitting batch {len(reports_batch)} -> {mid} + {len(reports_batch)-mid}")
            return (
                self.generate_recursive(reports_batch[:mid], max_new_tokens)
                + self.generate_recursive(reports_batch[mid:], max_new_tokens)
            )
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower() and len(reports_batch) > 1:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                mid = len(reports_batch) // 2
                return (
                    self.generate_recursive(reports_batch[:mid], max_new_tokens)
                    + self.generate_recursive(reports_batch[mid:], max_new_tokens)
                )
            raise

    def extract_batch(self, reports_batch):
        first_pass = self.generate_recursive(
            reports_batch,
            FAST_MAX_NEW_TOKENS,
        )

        results = []

        for report, generated in first_pass:
            rid = clean(report.get("report_id"))
            needs_retry = generated["hit_ceiling"]
            first_error = None

            if not needs_retry:
                try:
                    parsed = parse_json_object(generated["text"])
                    row = normalize_extracted_event(
                        parsed,
                        report,
                        generated["generated_tokens"],
                        FAST_MAX_NEW_TOKENS,
                        len(reports_batch),
                    )
                    results.append((report, row, None))
                    continue
                except Exception as exc:
                    first_error = exc
                    needs_retry = True

            # Quality safeguard: capped OR malformed -> rerun this report alone at 750.
            try:
                retry_generated = self.generate_batch(
                    [report],
                    RETRY_MAX_NEW_TOKENS,
                )[0]

                if retry_generated["hit_ceiling"]:
                    raise ValueError(
                        f"Generation still hit {RETRY_MAX_NEW_TOKENS}-token ceiling."
                    )

                parsed = parse_json_object(retry_generated["text"])
                row = normalize_extracted_event(
                    parsed,
                    report,
                    retry_generated["generated_tokens"],
                    RETRY_MAX_NEW_TOKENS,
                    1,
                )
                row["inference_audit"]["retry_reason"] = (
                    "first_attempt_hit_token_ceiling"
                    if generated["hit_ceiling"]
                    else f"first_attempt_parse_failure:{type(first_error).__name__}"
                )
                results.append((report, row, None))

            except Exception as retry_exc:
                results.append((
                    report,
                    None,
                    f"{type(retry_exc).__name__}: {retry_exc}",
                ))

        return results


In [ ]:

# ============================================================
# 6. EXTRACT ONLY NEW IDS + APPEND TO EXISTING JSONL
# ============================================================

newly_appended_ids = []
extraction_errors = []

# Keep the Qwen object for the category-fit pass so we do not reload 3B weights.
qwen_extractor = None

if new_reports_to_extract:
    qwen_extractor = IncrementalQwenExtractor(QWEN_MODEL)

    for start in range(0, len(new_reports_to_extract), EXTRACTION_BATCH_SIZE):
        batch = new_reports_to_extract[
            start:start + EXTRACTION_BATCH_SIZE
        ]

        print(
            f"NEW batch {start + 1}-"
            f"{min(start + len(batch), len(new_reports_to_extract))}"
            f"/{len(new_reports_to_extract)}:",
            [clean(r.get("report_id")) for r in batch],
            flush=True,
        )

        results = qwen_extractor.extract_batch(batch)

        # Re-read IDs before each durable append group, so a restarted/duplicated
        # cell can never append an ID that was already persisted.
        stored_ids_now = {
            clean(r.get("report_id"))
            for r in read_jsonl_rows(PREDICTED_EVENTS_PATH)
        }

        for report, row, error in results:
            rid = clean(report.get("report_id"))

            if rid in stored_ids_now:
                print("   SKIP already persisted:", rid)
                continue

            if error:
                print("   ERROR:", rid, error)
                extraction_errors.append({
                    "report_id": rid,
                    "error": error,
                })
                continue

            # THIS is the required behavior:
            # physically append NEW event at END of previous JSONL.
            append_event_durable(PREDICTED_EVENTS_PATH, row)
            stored_ids_now.add(rid)
            newly_appended_ids.append(rid)

            print("   APPENDED TO END:", rid, flush=True)

            if (
                SNAPSHOT_EVERY_NEW > 0
                and len(newly_appended_ids) % SNAPSHOT_EVERY_NEW == 0
            ):
                current_total = len(
                    unique_rows_preserve_first_position(
                        read_jsonl_rows(PREDICTED_EVENTS_PATH)
                    )
                )
                snap = make_full_snapshot(
                    PREDICTED_EVENTS_PATH,
                    current_total,
                )
                print("   snapshot:", snap)

else:
    print("No unseen report_id in the supplied new-data JSON. Nothing to extract.")

if extraction_errors:
    error_path = INFERENCE_DIR / "new_event_extraction_errors.json"
    error_path.write_text(
        json.dumps(extraction_errors, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Extraction errors:", error_path)

# Reload the actual combined file from Drive.
combined_physical_rows = read_jsonl_rows(PREDICTED_EVENTS_PATH)
combined_events = unique_rows_preserve_first_position(combined_physical_rows)
combined_ids = [clean(r["report_id"]) for r in combined_events]

print("=" * 72)
print("OLD unique rows before run:", len(existing_unique_before_run))
print("NEW rows appended this run:", len(newly_appended_ids))
print("COMBINED unique rows now:", len(combined_events))
print("Physical JSONL lines now:", len(combined_physical_rows))
print("Persistent file:", PREDICTED_EVENTS_PATH)
print("=" * 72)

# Strict append-order check for successfully appended IDs.
if newly_appended_ids:
    tail_ids = [
        clean(r.get("report_id"))
        for r in combined_physical_rows[-len(newly_appended_ids):]
    ]
    assert tail_ids == newly_appended_ids, (
        "Append-order check failed: new rows are not the physical tail."
    )
    print("APPEND ORDER CHECK: PASS")

missing_new_ids = [
    clean(r.get("report_id"))
    for r in new_reports_to_extract
    if clean(r.get("report_id")) not in set(combined_ids)
]

print("New reports still missing after extraction:", len(missing_new_ids))
if missing_new_ids:
    print("Missing IDs:", missing_new_ids[:30])


In [ ]:

# ============================================================
# 7. LOAD TAXONOMY + BUILD STRICT CLASS-FIT JOBS
# ============================================================

def resolve_if_missing(path, patterns):
    path = Path(path)
    if path.exists():
        return path
    candidates = []
    for pattern in patterns:
        candidates.extend(
            Path("/content/drive/MyDrive").rglob(pattern)
        )
    candidates = sorted(
        {p for p in candidates if p.is_file()},
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {path.name}. Edit TAXONOMY_PATH."
        )
    print("Auto-resolved taxonomy:", candidates[0])
    return candidates[0]

TAXONOMY_PATH = resolve_if_missing(
    TAXONOMY_PATH,
    ["AI_Harm_Map_Taxonomy_Schema_vSHARED*.xlsx"],
)

taxonomy_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name=TAXONOMY_SHEET,
).fillna("")
taxonomy_rows = taxonomy_df.to_dict("records")

def canonical_taxonomy_id(value):
    if value is None:
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return ""
        return f"{float(value):.10f}".rstrip("0").rstrip(".")
    return clean(value)

def classification_candidates(value):
    if value in (None, "", []):
        return []
    if isinstance(value, dict):
        return [copy.deepcopy(value)]
    if isinstance(value, list):
        return [copy.deepcopy(x) for x in value if x not in (None, "", [])]
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            return classification_candidates(json.loads(text))
        except Exception:
            pass

        parts = [p.strip() for p in text.split(";") if p.strip()]
        if len(parts) > 1:
            parsed = []
            all_json = True
            for part in parts:
                try:
                    parsed.extend(
                        classification_candidates(json.loads(part))
                    )
                except Exception:
                    all_json = False
                    break
            if all_json:
                return parsed

        return [text]

    return [copy.deepcopy(value)]

def candidate_id(candidate):
    if isinstance(candidate, dict):
        lower = {
            str(k).strip().casefold(): v
            for k, v in candidate.items()
        }
        for key in [
            "subcategory_id",
            "subcategory id",
            "#",
            "id",
            "taxonomy_id",
            "taxonomy id",
        ]:
            if key in lower and clean(lower[key]):
                return canonical_taxonomy_id(lower[key])

    if isinstance(candidate, (str, int, float)):
        text = canonical_taxonomy_id(candidate)
        if re.fullmatch(r"\d+(?:\.\d+)?", text):
            return text

    return ""

def candidate_name(candidate):
    if isinstance(candidate, dict):
        lower = {
            str(k).strip().casefold(): v
            for k, v in candidate.items()
        }
        for key in [
            "subcategory",
            "category",
            "harm_category",
            "harm category",
        ]:
            if key in lower and clean(lower[key]):
                return clean(lower[key])

    if isinstance(candidate, str):
        return candidate.strip()

    return ""

taxonomy_by_id = {}
taxonomy_by_name = {}

for row in taxonomy_rows:
    tid = canonical_taxonomy_id(row.get("#"))
    name = normalize(row.get("Subcategory"))
    if tid:
        taxonomy_by_id[tid] = row
    if name:
        taxonomy_by_name[name] = row

def match_taxonomy_row(candidate):
    tid = candidate_id(candidate)
    if tid and tid in taxonomy_by_id:
        return taxonomy_by_id[tid]

    name = normalize(candidate_name(candidate))
    if name and name in taxonomy_by_name:
        return taxonomy_by_name[name]

    if isinstance(candidate, str):
        text_id = canonical_taxonomy_id(candidate)
        if text_id in taxonomy_by_id:
            return taxonomy_by_id[text_id]
        if normalize(candidate) in taxonomy_by_name:
            return taxonomy_by_name[normalize(candidate)]

    return None

FIT_SYSTEM_PROMPT = r'''
You are a strict verifier of an already-assigned AI-harm taxonomy class.

You are NOT a classifier.

You MUST NOT choose, suggest, infer, rewrite, normalize, or replace a class.

You receive exactly ONE candidate class already assigned upstream.
Your only task is to decide whether THIS class fits THIS extracted event.

Use the authoritative Short definition and Inclusion test (proximate cause)
as the main criteria.

Return:
- "fit": event substantively satisfies this exact class.
- "not_fit": event clearly does not satisfy this exact class.
- "uncertain": evidence is insufficient for a safe decision.

Do not mark fit merely because the class is broadly related to AI harm.

Return JSON ONLY:
{
  "status": "fit",
  "confidence": 0.0,
  "reason": "brief evidence-grounded reason"
}

status must be exactly fit, not_fit, or uncertain.
reason must be at most 35 words.
'''

def verifier_event_view(event):
    return {
        "event_type": event.get("event_type", ""),
        "ai_system": event.get("ai_system", ""),
        "organization": event.get("organization", ""),
        "affected_group": event.get("affected_group", ""),
        "action": event.get("action", ""),
        "consequence": event.get("consequence", ""),
        "location": event.get("location", ""),
        "event_date": event.get("event_date", ""),
        "original_evidence_span": event.get("original_evidence_span", ""),
        "translated_evidence_span": event.get("translated_evidence_span", ""),
    }

def build_fit_prompt(event, candidate, row):
    payload = {
        "event": verifier_event_view(event),
        "candidate_classification": candidate,
        "authoritative_taxonomy_row": {
            "#": row.get("#", ""),
            "Branch": row.get("Branch", ""),
            "Subcategory": row.get("Subcategory", ""),
            "Short definition": row.get("Short definition", ""),
            "Inclusion test (proximate cause)": row.get(
                "Inclusion test (proximate cause)", ""
            ),
            "Protected interest": row.get("Protected interest", ""),
            "Event type": row.get("Event type", ""),
            "Occurrence status admitted": row.get(
                "Occurrence status admitted", ""
            ),
        },
    }
    return (
        "Does THIS supplied class fit THIS event? Evaluate only that candidate.\n\n"
        + json.dumps(payload, ensure_ascii=False, indent=2)
    )

verification_jobs = []

for event in combined_events:
    for idx, candidate in enumerate(
        classification_candidates(event.get("harm_category"))
    ):
        verification_jobs.append({
            "job_id": f"{clean(event['report_id'])}::{idx}",
            "report_id": clean(event["report_id"]),
            "candidate_index": idx,
            "candidate": candidate,
            "taxonomy_row": match_taxonomy_row(candidate),
            "event": event,
        })

print("Combined events:", len(combined_events))
print("Candidate categories:", len(verification_jobs))
print(
    "Matched to Excel:",
    sum(j["taxonomy_row"] is not None for j in verification_jobs),
)


In [ ]:

# ============================================================
# 8. RESUMABLE BATCHED CLASS-FIT PASS
# Reuses the already-loaded Qwen from extraction when possible.
# ============================================================

def load_audit(path):
    by_job = {}
    if not Path(path).exists():
        return by_job
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                if row.get("job_id"):
                    by_job[row["job_id"]] = row
            except Exception:
                continue
    return by_job

def append_audit(row):
    with VERIFY_AUDIT_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass

audit_by_job = load_audit(VERIFY_AUDIT_PATH)

# No taxonomy match => uncertain, never guess a class mapping.
for job in verification_jobs:
    if job["job_id"] in audit_by_job:
        continue
    if job["taxonomy_row"] is None:
        row = {
            "job_id": job["job_id"],
            "report_id": job["report_id"],
            "candidate_index": job["candidate_index"],
            "candidate": job["candidate"],
            "status": "uncertain",
            "confidence": 0.0,
            "reason": "Candidate could not be matched to an authoritative Excel taxonomy row.",
            "taxonomy_match": False,
        }
        append_audit(row)
        audit_by_job[job["job_id"]] = row

pending_jobs = [
    j for j in verification_jobs
    if j["job_id"] not in audit_by_job
]

print("Already audited:", len(audit_by_job))
print("Pending Qwen class-fit checks:", len(pending_jobs))

if RUN_CLASS_FIT and pending_jobs:
    # Reuse extraction Qwen weights if they are already in memory.
    if qwen_extractor is not None:
        verify_tokenizer = qwen_extractor.tokenizer
        verify_model = qwen_extractor.model
        print("Reusing Qwen already loaded for extraction.")
    else:
        from transformers import AutoTokenizer, AutoModelForCausalLM

        verify_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
        verify_tokenizer.padding_side = "left"
        if verify_tokenizer.pad_token_id is None:
            verify_tokenizer.pad_token = verify_tokenizer.eos_token

        kwargs = {
            "device_map": "auto",
            "torch_dtype": (
                torch.float16
                if torch.cuda.is_available()
                else torch.float32
            ),
        }
        if torch.cuda.is_available():
            kwargs["attn_implementation"] = "sdpa"

        try:
            verify_model = AutoModelForCausalLM.from_pretrained(
                QWEN_MODEL,
                **kwargs,
            )
        except Exception:
            kwargs.pop("attn_implementation", None)
            verify_model = AutoModelForCausalLM.from_pretrained(
                QWEN_MODEL,
                **kwargs,
            )

        verify_model.eval()

    def generate_verify_batch(batch):
        prompts = []
        for job in batch:
            messages = [
                {"role": "system", "content": FIT_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": build_fit_prompt(
                        job["event"],
                        job["candidate"],
                        job["taxonomy_row"],
                    ),
                },
            ]
            prompts.append(
                verify_tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )

        encoded = verify_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=VERIFY_MAX_INPUT_TOKENS,
        )
        device = next(verify_model.parameters()).device
        encoded = {k: v.to(device) for k, v in encoded.items()}
        width = encoded["input_ids"].shape[1]

        with torch.inference_mode():
            generated = verify_model.generate(
                **encoded,
                max_new_tokens=VERIFY_MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                eos_token_id=verify_tokenizer.eos_token_id,
                pad_token_id=verify_tokenizer.pad_token_id,
            )

        texts = [
            verify_tokenizer.decode(
                seq[width:],
                skip_special_tokens=True,
            )
            for seq in generated
        ]

        del encoded, generated
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return texts

    def verify_recursive(batch):
        try:
            return list(zip(batch, generate_verify_batch(batch)))
        except torch.cuda.OutOfMemoryError:
            if len(batch) == 1:
                raise
            torch.cuda.empty_cache()
            mid = len(batch) // 2
            return (
                verify_recursive(batch[:mid])
                + verify_recursive(batch[mid:])
            )

    for start in range(0, len(pending_jobs), VERIFY_BATCH_SIZE):
        batch = pending_jobs[start:start + VERIFY_BATCH_SIZE]

        for job, text in verify_recursive(batch):
            try:
                parsed = parse_json_object(text)
                status = clean(parsed.get("status")).casefold()
                if status not in {"fit", "not_fit", "uncertain"}:
                    status = "uncertain"

                try:
                    confidence = float(
                        np.clip(
                            float(parsed.get("confidence", 0.0)),
                            0.0,
                            1.0,
                        )
                    )
                except Exception:
                    confidence = 0.0

                reason = clean(parsed.get("reason"))
                if not reason:
                    reason = "No valid verifier reason returned."

            except Exception as exc:
                status = "uncertain"
                confidence = 0.0
                reason = f"Verifier parse failure: {type(exc).__name__}"

            row = {
                "job_id": job["job_id"],
                "report_id": job["report_id"],
                "candidate_index": job["candidate_index"],
                "candidate": job["candidate"],
                "status": status,
                "confidence": confidence,
                "reason": reason,
                "taxonomy_match": True,
                "taxonomy_id": canonical_taxonomy_id(
                    job["taxonomy_row"].get("#")
                ),
                "taxonomy_subcategory": clean(
                    job["taxonomy_row"].get("Subcategory")
                ),
            }

            append_audit(row)
            audit_by_job[job["job_id"]] = row

        print(
            "Class-fit:",
            min(start + len(batch), len(pending_jobs)),
            "/",
            len(pending_jobs),
        )

# Free Qwen before graph model.
if qwen_extractor is not None:
    try:
        del qwen_extractor.model
        del qwen_extractor.tokenizer
    except Exception:
        pass
    qwen_extractor = None

for name in ["verify_model", "verify_tokenizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Class-fit audit:", VERIFY_AUDIT_PATH)


In [ ]:

# ============================================================
# 9. ATTACH CLASS-FIT AUDIT TO ALL OLD + NEW EVENTS
# ============================================================

events_by_id = {}
verified_for_retraining = []
fit_summary_rows = []

for event in combined_events:
    out = copy.deepcopy(event)
    supplied = classification_candidates(event.get("harm_category"))

    candidate_audits = []
    verified = []

    for idx, candidate in enumerate(supplied):
        audit = audit_by_job.get(
            f"{clean(event['report_id'])}::{idx}",
            {
                "status": "uncertain",
                "confidence": 0.0,
                "reason": "Verification not available.",
            },
        )

        item = {
            "candidate": copy.deepcopy(candidate),
            "status": audit["status"],
            "confidence": audit["confidence"],
            "reason": audit["reason"],
            "taxonomy_id": audit.get("taxonomy_id"),
            "taxonomy_subcategory": audit.get(
                "taxonomy_subcategory"
            ),
        }
        candidate_audits.append(item)

        if audit["status"] in ACCEPTED_CLASS_STATUSES:
            verified.append(copy.deepcopy(candidate))

        fit_summary_rows.append({
            "report_id": clean(event["report_id"]),
            "is_new_this_run": (
                clean(event["report_id"])
                in set(newly_appended_ids)
            ),
            "candidate_index": idx,
            "status": audit["status"],
            "confidence": audit["confidence"],
            "taxonomy_id": audit.get("taxonomy_id"),
            "taxonomy_subcategory": audit.get(
                "taxonomy_subcategory"
            ),
            "reason": audit["reason"],
        })

    out["supplied_harm_category"] = copy.deepcopy(
        event.get("harm_category")
    )
    out["verified_harm_category"] = verified
    out["harm_category_verification"] = {
        "method": "candidate_fit_check_only",
        "can_reclassify": False,
        "accepted_statuses": sorted(
            ACCEPTED_CLASS_STATUSES
        ),
        "candidates": candidate_audits,
    }
    out["incremental_inference"] = {
        "is_new_this_run": (
            clean(event["report_id"])
            in set(newly_appended_ids)
        )
    }

    events_by_id[clean(event["report_id"])] = out

    retrain_row = copy.deepcopy(out)
    retrain_row["harm_category"] = copy.deepcopy(verified)
    verified_for_retraining.append(retrain_row)

fit_summary_df = pd.DataFrame(fit_summary_rows)
if len(fit_summary_df):
    display(
        fit_summary_df["status"]
        .value_counts(dropna=False)
        .rename("count")
    )

fit_summary_df.to_csv(
    INFERENCE_DIR / "harm_category_fit_summary.csv",
    index=False,
)

VERIFIED_RETRAIN_PATH = (
    INFERENCE_DIR / "events_ALL_verified_for_retraining.jsonl"
)
with VERIFIED_RETRAIN_PATH.open("w", encoding="utf-8") as f:
    for row in verified_for_retraining:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Verified-for-retraining file:", VERIFIED_RETRAIN_PATH)


In [ ]:
# ============================================================
# 10. LOAD EXISTING PHTKG CHECKPOINT — NO RETRAINING
# ============================================================

def resolve_checkpoint(path):
    path = Path(path)
    if path.exists():
        return path

    candidates = sorted(
        Path("/content/drive/MyDrive").rglob("predicted_phtkg.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            "predicted_phtkg.pt not found. Edit PHTKG_CHECKPOINT_PATH."
        )
    print("Auto-resolved checkpoint:", candidates[0])
    return candidates[0]

PHTKG_CHECKPOINT_PATH = resolve_checkpoint(PHTKG_CHECKPOINT_PATH)

DEFAULT_ROLES = [
    "organization",
    "ai_system",
    "affected_group",
    "action",
    "harm_category",
    "consequence",
    "location",
]
EMPTY = "__EMPTY__"

checkpoint = torch.load(
    PHTKG_CHECKPOINT_PATH, map_location="cpu", weights_only=False
)

ROLES = checkpoint.get("roles", DEFAULT_ROLES)
PHTKG_CONFIG = dict(checkpoint.get("phtkg_config", {}))
PHTKG_CONFIG.setdefault("EMBEDDING_DIM", 64)
PHTKG_CONFIG.setdefault("LAYERS", 2)
PHTKG_CONFIG.setdefault("DROPOUT", 0.10)

class PHTKG(nn.Module):
    def __init__(self, total_nodes, config):
        super().__init__()
        cfg = dict(config)
        d = int(cfg["EMBEDDING_DIM"])

        self.layers = int(cfg["LAYERS"])
        self.dimension = d

        self.entity_embeddings = nn.Embedding(total_nodes, d)
        self.role_embeddings = nn.Parameter(torch.randn(len(ROLES), d) * 0.02)
        self.event_seed = nn.Parameter(torch.randn(d) * 0.02)

        self.time_linear_weight = nn.Parameter(torch.randn(d))
        self.time_linear_bias = nn.Parameter(torch.zeros(d))
        self.time_periodic_weight = nn.Parameter(torch.randn(d))
        self.time_periodic_bias = nn.Parameter(torch.zeros(d))
        self.missing_time = nn.Parameter(torch.randn(d) * 0.02)

        self.provenance_message = nn.Sequential(
            nn.Linear(1, d),
            nn.Tanh(),
        )

        self.entity_key = nn.Linear(d, d, bias=False)
        self.entity_value = nn.Linear(d, d, bias=False)
        self.event_query = nn.Linear(d, d, bias=False)
        self.role_bias = nn.Parameter(torch.zeros(len(ROLES)))

        self.dropout = nn.Dropout(float(cfg["DROPOUT"]))
        self.event_update = nn.GRUCell(d, d)
        self.entity_update = nn.GRUCell(d, d)

        self.event_to_entity = nn.ModuleList(
            [nn.Linear(d, d, bias=False) for _ in ROLES]
        )

        self.temporal_decay = nn.Parameter(torch.zeros(len(ROLES)))

        self.scorer = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, 1),
        )

        self.next_year_head = nn.Sequential(
            nn.Linear(2 * d, d),
            nn.GELU(),
            nn.Linear(d, 1),
        )

    def encode_time(self, year_normalized, device):
        y = torch.tensor([[year_normalized]], dtype=torch.float32, device=device)
        return (
            y * self.time_linear_weight
            + self.time_linear_bias
            + torch.sin(y * self.time_periodic_weight + self.time_periodic_bias)
        ).squeeze(0)

    def forward(
        self,
        global_ids,
        years,
        known,
        provenance,
        time_mean,
        time_known,
        entity_state_init=None,
    ):
        event_count = global_ids.shape[0]

        entity_state = (
            self.entity_embeddings.weight
            if entity_state_init is None
            else entity_state_init
        )

        event_state = self.event_seed.unsqueeze(0).expand(event_count, -1)

        years_column = years.unsqueeze(-1)
        known_column = known.unsqueeze(-1)

        observed_time = (
            years_column * self.time_linear_weight
            + self.time_linear_bias
            + torch.sin(
                years_column * self.time_periodic_weight + self.time_periodic_bias
            )
        )

        time_message = (
            known_column * observed_time + (1 - known_column) * self.missing_time
        )

        provenance_message = self.provenance_message(provenance.unsqueeze(-1))
        provenance_gate = torch.sigmoid(provenance).unsqueeze(-1)

        for _ in range(self.layers):
            incident = entity_state[global_ids] + self.role_embeddings.unsqueeze(0)

            keys = self.entity_key(incident)
            values = self.entity_value(incident)
            query = self.event_query(event_state).unsqueeze(1)

            logits = (
                (query * keys).sum(-1) / math.sqrt(self.dimension)
                + self.role_bias.unsqueeze(0)
            )

            attention = F.softmax(logits, dim=1)

            entity_message = self.dropout(
                (attention.unsqueeze(-1) * values).sum(1)
            )

            event_state = self.event_update(
                entity_message + time_message + provenance_message,
                event_state,
            )

            aggregate = torch.zeros_like(entity_state)
            denominator = torch.zeros(
                entity_state.shape[0], 1, device=entity_state.device
            )

            for role_index in range(len(ROLES)):
                ids = global_ids[:, role_index]

                temporal_known = known * time_known[ids]

                gap = torch.abs(years - time_mean[ids])

                decay = F.softplus(self.temporal_decay[role_index])

                temporal_weight = (
                    temporal_known * torch.exp(-decay * gap) + (1 - temporal_known)
                )

                weight = temporal_weight * provenance_gate.squeeze(-1)

                message = (
                    self.event_to_entity[role_index](event_state)
                    * weight.unsqueeze(-1)
                )

                aggregate.index_add_(0, ids, message)
                denominator.index_add_(0, ids, weight.unsqueeze(-1))

            entity_input = self.dropout(
                aggregate / torch.clamp(denominator, min=1.0)
            )

            proposed_state = self.entity_update(entity_input, entity_state)

            entity_state = torch.where(denominator > 0, proposed_state, entity_state)

        embedding = F.normalize(event_state, dim=-1)

        return {
            "embedding": embedding,
            "score": self.scorer(embedding).squeeze(-1),
            "entity_state": entity_state,
        }

    def predict_recurrence(self, entity_ids, entity_state, target_year_normalized):
        time_query = self.encode_time(target_year_normalized, entity_state.device)

        time_query = time_query.unsqueeze(0).expand(len(entity_ids), -1)

        logits = self.next_year_head(
            torch.cat([entity_state[entity_ids], time_query], dim=-1)
        ).squeeze(-1)

        return torch.sigmoid(logits)

state_dict = checkpoint["model_state_dict"]
total_nodes = int(state_dict["entity_embeddings.weight"].shape[0])

predicted_model = PHTKG(total_nodes, PHTKG_CONFIG).to(DEVICE)

predicted_model.load_state_dict(state_dict, strict=True)
predicted_model.eval()

vocab = checkpoint["vocab"]
offsets = checkpoint["offsets"]
year_min = int(checkpoint["year_min"])
year_max = int(checkpoint["year_max"])

checkpoint_final_state = checkpoint.get(
    "final_entity_state",
    predicted_model.entity_embeddings.weight.detach().cpu(),
).to(DEVICE)

print("Checkpoint loaded:", PHTKG_CHECKPOINT_PATH)
print("Nodes:", total_nodes)
print("Embedding dim:", predicted_model.dimension)
print("Training years:", year_min, "->", year_max)

In [ ]:
# ============================================================
# 11. CHECKPOINT-COMPATIBLE EVENT ENCODING
# IMPORTANT: reconstruct checkpoint time context from OLD corpus only,
#            not from newly appended inference events.
# ============================================================

VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

def graph_text(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return clean(value)

def year_from(value):
    match = re.search(r"\b(?:19|20)\d{2}\b", clean(value))
    return int(match.group(0)) if match else 0

def prediction_provenance(event):
    try:
        return float(np.clip(float(event.get("confidence", 0.5)), 0.0, 1.0))
    except Exception:
        return 0.5

def role_values(event):
    # Original supplied harm_category is intentional for checkpoint compatibility.
    return {
        "organization": clean(event.get("organization")) or EMPTY,
        "ai_system": clean(event.get("ai_system")) or EMPTY,
        "affected_group": clean(event.get("affected_group")) or EMPTY,
        "action": clean(event.get("action")) or EMPTY,
        "harm_category": graph_text(event.get("harm_category")) or EMPTY,
        "consequence": clean(event.get("consequence")) or EMPTY,
        "location": clean(event.get("location")) or EMPTY,
    }

def make_edges(event_rows):
    return [
        {
            "event_id": clean(event["report_id"]),
            "values": role_values(event),
            "year": year_from(event.get("event_date")),
            "provenance": prediction_provenance(event),
        }
        for event in event_rows
    ]

def checkpoint_training_split(edges):
    # Matches the full-model notebook's chronological split.
    if len(edges) < 8:
        shuffled = list(edges)
        random.Random(SEED).shuffle(shuffled)
        return shuffled

    dated = sorted(
        [edge for edge in edges if edge["year"] > 0],
        key=lambda edge: (edge["year"], edge["event_id"]),
    )

    undated = [edge for edge in edges if edge["year"] == 0]

    if len(dated) < 8:
        dated = list(edges)
        random.Random(SEED).shuffle(dated)
        undated = []

    test_n = max(1, round(len(dated) * TEST_FRACTION))
    val_n = max(1, round(len(dated) * VALIDATION_FRACTION))

    train_end = len(dated) - val_n - test_n

    return dated[:train_end] + undated

def encode_edges(edges):
    rows = []
    unknown = Counter()
    totals = Counter()

    for edge in edges:
        ids = []

        for role in ROLES:
            value = normalize(edge["values"][role]) or EMPTY

            role_vocab = vocab[role]
            totals[role] += 1

            if value not in role_vocab:
                unknown[role] += 1

            local_id = role_vocab.get(value, 0)

            ids.append(offsets[role] + local_id)

        known = float(edge["year"] > 0)

        year_norm = (
            (edge["year"] - year_min) / max(1, year_max - year_min)
            if edge["year"] > 0
            else 0.0
        )

        rows.append({
            "event_id": edge["event_id"],
            "global_ids": ids,
            "year": float(year_norm),
            "known": known,
            "raw_year": int(edge["year"]),
            "provenance": float(edge["provenance"]),
        })

    return rows, unknown, totals

def to_tensors(rows):
    return (
        torch.tensor([r["global_ids"] for r in rows], dtype=torch.long, device=DEVICE),
        torch.tensor([r["year"] for r in rows], dtype=torch.float32, device=DEVICE),
        torch.tensor([r["known"] for r in rows], dtype=torch.float32, device=DEVICE),
        torch.tensor([r["provenance"] for r in rows], dtype=torch.float32, device=DEVICE),
    )

combined_model_events = [
    events_by_id[clean(e["report_id"])] for e in combined_events
]

combined_edges = make_edges(combined_model_events)

combined_rows, unknown, totals = encode_edges(combined_edges)

# OLD events are the checkpoint-era corpus for time-context reconstruction.
old_model_events = [
    events_by_id[clean(e["report_id"])]
    for e in existing_unique_before_run
    if clean(e["report_id"]) in events_by_id
]

old_edges = make_edges(old_model_events)
old_train_edges = checkpoint_training_split(old_edges)
old_train_rows, _, _ = encode_edges(old_train_edges)

unknown_df = pd.DataFrame([
    {
        "role": role,
        "unknown_count": unknown[role],
        "total": totals[role],
        "unknown_rate": unknown[role] / max(1, totals[role]),
    }
    for role in ROLES
])

display(unknown_df)

print("Combined events for inference:", len(combined_rows))
print("Old checkpoint-era events:", len(old_model_events))
print("Newly appended events:", len(newly_appended_ids))

if len(existing_unique_before_run) == 0:
    print(
        "WARNING: No old extraction corpus existed before this run. "
        "Time-context reconstruction cannot reproduce the checkpoint training corpus."
    )

In [ ]:
# ============================================================
# 12. RECONSTRUCT FIXED CHECKPOINT TIME CONTEXT FROM OLD DATA
# ============================================================

def compute_time_context(rows):
    if not rows:
        return (
            torch.zeros(total_nodes, device=DEVICE),
            torch.zeros(total_nodes, device=DEVICE),
        )

    ids, years, knowns, provs = to_tensors(rows)

    sums = torch.zeros(total_nodes, device=DEVICE)
    counts = torch.zeros(total_nodes, device=DEVICE)

    for role_index in range(len(ROLES)):
        role_ids = ids[:, role_index]

        sums.index_add_(0, role_ids, years * knowns)
        counts.index_add_(0, role_ids, knowns)

    return (
        sums / torch.clamp(counts, min=1.0),
        (counts > 0).float(),
    )

if "time_mean" in checkpoint and "time_known" in checkpoint:
    time_mean = checkpoint["time_mean"].to(DEVICE)
    time_known = checkpoint["time_known"].to(DEVICE)
    print("Using time context stored directly in checkpoint.")
else:
    time_mean, time_known = compute_time_context(old_train_rows)
    print(
        "Checkpoint did not save time_mean/time_known; "
        "reconstructed them from the OLD pre-increment corpus."
    )

In [ ]:

# ============================================================
# 13. PHTKG CHECKPOINT INFERENCE ON ALL OLD + NEW EVENTS
# ============================================================

ids_tensor, years_tensor, known_tensor, provenance_tensor = to_tensors(
    combined_rows
)

with torch.inference_mode():
    phtkg_out = predicted_model(
        ids_tensor,
        years_tensor,
        known_tensor,
        provenance_tensor,
        time_mean,
        time_known,
        entity_state_init=checkpoint_final_state,
    )

embedding_matrix = (
    phtkg_out["embedding"]
    .detach()
    .cpu()
    .numpy()
)

embedding_matrix = (
    embedding_matrix
    / np.clip(
        np.linalg.norm(
            embedding_matrix,
            axis=1,
            keepdims=True,
        ),
        1e-12,
        None,
    )
)

phtkg_scores = (
    torch.sigmoid(
        phtkg_out["score"]
    )
    .detach()
    .cpu()
    .numpy()
)

with torch.inference_mode():
    yc = years_tensor.unsqueeze(-1)
    kc = known_tensor.unsqueeze(-1)

    observed_time = (
        yc
        * predicted_model.time_linear_weight
        + predicted_model.time_linear_bias
        + torch.sin(
            yc
            * predicted_model.time_periodic_weight
            + predicted_model.time_periodic_bias
        )
    )

    temporal_matrix = (
        kc * observed_time
        + (
            1 - kc
        )
        * predicted_model.missing_time
    ).detach().cpu().numpy()

event_ids = [
    row["event_id"]
    for row in combined_rows
]

print("Combined event embeddings:", embedding_matrix.shape)
print("Temporal representations:", temporal_matrix.shape)


In [ ]:
# ============================================================
# 14. BATCHED NEXT-YEAR RECURRENCE
# ============================================================

def global_id_if_known(role, value):
    value = normalize(value) or EMPTY
    if value not in vocab[role]:
        return None
    return offsets[role] + vocab[role][value]

event_lookup = {clean(e["report_id"]): e for e in combined_model_events}

recurrence_by_event = {
    rid: {"ai_system": None, "harm_category": None} for rid in event_ids
}

groups = defaultdict(list)

for rid in event_ids:
    event = event_lookup[rid]
    event_year = year_from(event.get("event_date"))
    base_year = event_year if event_year else year_max

    target_year = base_year + 1
    target_norm = (target_year - year_min) / max(1, year_max - year_min)

    for role in ["ai_system", "harm_category"]:
        value = (
            event.get("ai_system")
            if role == "ai_system"
            else graph_text(event.get("harm_category"))
        )

        gid = global_id_if_known(role, value)

        if gid is not None:
            groups[(role, target_year, float(target_norm))].append((rid, gid))

for (role, target_year, target_norm), members in groups.items():

    entity_ids = torch.tensor(
        [gid for _, gid in members], dtype=torch.long, device=DEVICE
    )

    with torch.inference_mode():
        probs = (
            predicted_model
            .predict_recurrence(entity_ids, checkpoint_final_state, target_norm)
            .detach()
            .cpu()
            .numpy()
        )

    for (rid, _), probability in zip(members, probs):
        recurrence_by_event[rid][role] = {
            "target_year": int(target_year),
            "probability": float(probability),
        }

print("Recurrence batches:", len(groups))

In [ ]:
# ============================================================
# 15. INCREMENTAL TEMPORAL EVOLUTION OF NEW EVENTS
# Start from checkpoint final entity state and apply only NEW events
# chronologically. This does not retrain model weights.
# ============================================================

new_event_id_set = set(newly_appended_ids)

new_rows_for_temporal = [
    row for row in combined_rows if row["event_id"] in new_event_id_set
]

def incremental_temporal_rollout(model, new_rows):
    if not new_rows:
        return pd.DataFrame(), {}, checkpoint_final_state

    buckets = defaultdict(list)

    for row in new_rows:
        key = row["raw_year"] if row["raw_year"] > 0 else None
        buckets[key].append(row)

    ordered = []

    if None in buckets:
        ordered.append((None, buckets[None]))

    for y in sorted(k for k in buckets if k is not None):
        ordered.append((y, buckets[y]))

    state = checkpoint_final_state.detach()
    snapshots = {}
    drift_rows = []

    with torch.inference_mode():
        for bucket_year, bucket_rows in ordered:
            ids, ys, ks, ps = to_tensors(bucket_rows)

            result = model(
                ids, ys, ks, ps, time_mean, time_known, entity_state_init=state
            )

            new_state = result["entity_state"]

            distance = torch.linalg.vector_norm(new_state - state, dim=1)

            drift_rows.append({
                "year": int(bucket_year) if bucket_year is not None else None,
                "events_added": len(bucket_rows),
                "mean_entity_state_drift": float(distance.mean().cpu()),
                "median_entity_state_drift": float(distance.median().cpu()),
                "max_entity_state_drift": float(distance.max().cpu()),
            })

            if bucket_year is not None:
                snapshots[int(bucket_year)] = new_state.detach().cpu()

            state = new_state

    return pd.DataFrame(drift_rows), snapshots, state

incremental_drift_df, incremental_snapshots, updated_entity_state = (
    incremental_temporal_rollout(predicted_model, new_rows_for_temporal)
)

incremental_drift_df.to_csv(
    INFERENCE_DIR / "incremental_new_event_entity_drift.csv", index=False
)

display(incremental_drift_df)

new_dated_years = [
    row["raw_year"] for row in new_rows_for_temporal if row["raw_year"] > 0
]

if new_dated_years and min(new_dated_years) < year_max:
    print(
        "NOTE: some newly supplied reports describe events dated before "
        f"checkpoint year_max={year_max}. Their embeddings are valid fixed-checkpoint "
        "inference, but the incremental state rollout is not a retroactive retraining."
    )

In [ ]:
# ============================================================
# 16. FAST COARSE-TO-FINE KMEANS ON COMBINED OLD + NEW EMBEDDINGS
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

n_events = len(embedding_matrix)

if n_events < 3:
    n_patterns = 1
    pattern_labels = np.zeros(n_events, dtype=int)
    cluster_centers = embedding_matrix.mean(axis=0, keepdims=True)
    clustering_diagnostics = pd.DataFrame()

else:
    max_k = min(n_events - 1, max(2, n_events // MIN_PATTERN_SIZE))

    coarse_count = min(COARSE_K_POINTS, max(1, max_k - 1))

    coarse_ks = sorted(
        set(np.linspace(2, max_k, num=coarse_count, dtype=int).tolist())
    )

    results = {}

    def fit_k(k):
        if k in results:
            return results[k]

        km = KMeans(
            n_clusters=int(k),
            random_state=SEED,
            n_init=KMEANS_N_INIT,
            algorithm="lloyd",
        )

        labels = km.fit_predict(embedding_matrix)
        counts = np.bincount(labels, minlength=k)

        sil = None
        if len(np.unique(labels)) >= 2:
            sil = float(
                silhouette_score(embedding_matrix, labels, metric="euclidean")
            )

        results[int(k)] = {
            "labels": labels,
            "centers": km.cluster_centers_,
            "inertia": float(km.inertia_),
            "silhouette": sil,
            "min_cluster_size": int(counts.min()),
            "max_cluster_size": int(counts.max()),
            "valid_min_size": int(counts.min()) >= MIN_PATTERN_SIZE,
        }

        return results[int(k)]

    print("Coarse K search:", coarse_ks)

    for k in coarse_ks:
        fit_k(k)

    valid = [
        k for k in coarse_ks
        if results[k]["silhouette"] is not None and results[k]["valid_min_size"]
    ]

    if not valid:
        valid = [k for k in coarse_ks if results[k]["silhouette"] is not None]

    coarse_best = max(valid, key=lambda k: results[k]["silhouette"])

    pos = coarse_ks.index(coarse_best)
    left = coarse_ks[max(0, pos - 1)]
    right = coarse_ks[min(len(coarse_ks) - 1, pos + 1)]

    print(f"Fine K search: {left}...{right}")

    for k in range(left, right + 1):
        fit_k(k)

    valid = [
        k for k in results
        if results[k]["silhouette"] is not None and results[k]["valid_min_size"]
    ]

    if not valid:
        valid = [k for k in results if results[k]["silhouette"] is not None]

    chosen_k = max(valid, key=lambda k: results[k]["silhouette"])

    n_patterns = int(chosen_k)
    pattern_labels = results[chosen_k]["labels"]
    cluster_centers = results[chosen_k]["centers"]

    clustering_diagnostics = pd.DataFrame([
        {
            "k": k,
            "silhouette": results[k]["silhouette"],
            "inertia": results[k]["inertia"],
            "min_cluster_size": results[k]["min_cluster_size"],
            "max_cluster_size": results[k]["max_cluster_size"],
            "valid_min_size": results[k]["valid_min_size"],
            "chosen": k == chosen_k,
        }
        for k in sorted(results)
    ])

    clustering_diagnostics.to_csv(
        INFERENCE_DIR / "clustering_k_search_ALL.csv", index=False
    )

    print("K values evaluated:", len(results))
    print("Chosen patterns:", n_patterns)
    print("Silhouette:", results[chosen_k]["silhouette"])

display(clustering_diagnostics)

In [ ]:

# ============================================================
# 17. SIMILAR EVENTS + PATTERN PROBABILITY + NOVELTY
# ============================================================

centers_norm = (
    cluster_centers
    / np.clip(
        np.linalg.norm(
            cluster_centers,
            axis=1,
            keepdims=True,
        ),
        1e-12,
        None,
    )
)

pattern_similarity = (embedding_matrix@ centers_norm.T)

pattern_logits = (pattern_similarity/ 0.10)

pattern_logits -= (pattern_logits.max(axis=1,keepdims=True,))

pattern_probs = np.exp(pattern_logits)

pattern_probs /= np.clip(pattern_probs.sum(axis=1,keepdims=True,),1e-12,None,)

# ~500-1000 events: exact all-pairs cosine is cheap and deterministic.
similarity_matrix = (embedding_matrix@ embedding_matrix.T)

similar_events = {}

for i, rid in enumerate(event_ids):
    ranking = np.argsort( -similarity_matrix[i])

    neighbors = []

    for j in ranking:
        if j == i:
            continue

        neighbors.append({
            "report_id": event_ids[j],
            "similarity": round(
                float(similarity_matrix[i,j,])  6,),
        })

        if len(neighbors) >= TOP_K_SIMILAR:
            break

    similar_events[rid] = neighbors

best_center_similarity = (
    pattern_similarity.max(
        axis=1
    )
)

novelty_scores = np.clip(1.0- ((best_center_similarity + 1.0)/ 2.0),0.0,1.0,
)

print(
    "Pattern probability matrix:",
    pattern_probs.shape,
)


In [ ]:
# ============================================================
# 18. PATTERN TEMPORAL EVOLUTION OVER THE COMBINED CORPUS
# ============================================================

def frequency_trend_for(years_list):
    counts = Counter(years_list)
    years_sorted = sorted(counts)

    if len(years_sorted) < 2:
        return {
            "counts_by_year": {str(k): int(v) for k, v in sorted(counts.items())},
            "slope_events_per_year": None,
            "trend": "insufficient_data",
        }

    xs = np.asarray(years_sorted, dtype=float)
    ys = np.asarray([counts[y] for y in years_sorted], dtype=float)
    slope = float(np.polyfit(xs, ys, 1)[0])

    if slope > FREQUENCY_TREND_THRESHOLD:
        trend = "increasing"
    elif slope < -FREQUENCY_TREND_THRESHOLD:
        trend = "decreasing"
    else:
        trend = "stable"

    return {
        "counts_by_year": {str(k): int(v) for k, v in sorted(counts.items())},
        "slope_events_per_year": slope,
        "trend": trend,
    }

pattern_evolution = {}
pattern_summary_rows = []

for p in range(n_patterns):
    member_indices = np.flatnonzero(pattern_labels == p)

    members = []
    years_list = []

    for i in member_indices:
        rid = event_ids[i]
        y = year_from(event_lookup[rid].get("event_date"))

        members.append({
            "report_id": rid,
            "year": int(y) if y else None,
            "is_new_this_run": rid in new_event_id_set,
        })

        if y:
            years_list.append(int(y))

    pattern_name = f"pattern_{p + 1}"
    trend = frequency_trend_for(years_list)

    pattern_evolution[pattern_name] = {
        "event_count": int(len(member_indices)),
        "new_events_this_run": int(
            sum(rid in new_event_id_set for rid in [event_ids[i] for i in member_indices])
        ),
        "years": sorted(years_list),
        "events": members,
        "frequency_trend": trend,
    }

    pattern_summary_rows.append({
        "pattern": pattern_name,
        "event_count": len(member_indices),
        "new_events_this_run": pattern_evolution[pattern_name]["new_events_this_run"],
        "first_year": min(years_list) if years_list else None,
        "last_year": max(years_list) if years_list else None,
        "slope_events_per_year": trend["slope_events_per_year"],
        "trend": trend["trend"],
    })

pattern_summary_df = pd.DataFrame(pattern_summary_rows)

pattern_summary_df.to_csv(INFERENCE_DIR / "pattern_evolution_ALL.csv", index=False)

display(pattern_summary_df.sort_values("event_count", ascending=False))

In [ ]:
# ============================================================
# 19. BUILD FINAL ALL-TOGETHER OUTPUT
# Order = existing rows first + new appended rows at end.
# ============================================================

final_events = []

for i, rid in enumerate(event_ids):
    event = copy.deepcopy(events_by_id[rid])

    best_pattern_idx = int(np.argmax(pattern_probs[i]))
    pattern_name = f"pattern_{best_pattern_idx + 1}"
    best_probability = float(pattern_probs[i, best_pattern_idx])

    recurrence = recurrence_by_event.get(rid, {})

    event.pop("graph_representation", None)
    event.pop("counterpart", None)

    event["learned_representation"] = {
        "event_embedding": [round(float(x), 6) for x in embedding_matrix[i]],
        "phtkg_event_score": round(float(phtkg_scores[i]), 6),
        "temporal_representation": [round(float(x), 6) for x in temporal_matrix[i]],
        "pattern_assignment": pattern_name,
        "pattern_probability": round(best_probability, 6),
        "novelty_score": round(float(novelty_scores[i]), 6),
        "similar_events": similar_events[rid],
        "ai_system_recurrence": recurrence.get("ai_system"),
        "harm_category_recurrence": recurrence.get("harm_category"),
        "pattern_evolution": pattern_evolution[pattern_name],
        "checkpoint_category_policy": (
            "Existing PHTKG checkpoint uses the original supplied "
            "harm_category. verified_harm_category is post-extraction "
            "fit auditing and is saved separately."
        ),
    }

    final_events.append(event)

# Atomic rebuild of the ENRICHED output is okay:
# raw predicted_events.jsonl remains append-only and untouched.
temp_jsonl = FINAL_ENRICHED_JSONL.with_suffix(".tmp")

with temp_jsonl.open("w", encoding="utf-8") as f:
    for event in final_events:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")

temp_jsonl.replace(FINAL_ENRICHED_JSONL)

FINAL_ENRICHED_JSON.write_text(
    json.dumps(final_events, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

assert [clean(e["report_id"]) for e in final_events] == event_ids

if newly_appended_ids:
    assert event_ids[-len(newly_appended_ids):] == newly_appended_ids

print("FINAL combined events:", len(final_events))
print("Raw append-only history:", PREDICTED_EVENTS_PATH)
print("Final enriched JSONL:", FINAL_ENRICHED_JSONL)
print("Final enriched JSON:", FINAL_ENRICHED_JSON)
print("New IDs at final tail:", newly_appended_ids)

In [ ]:
# ============================================================
# 20. MANIFEST / RUN SUMMARY
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

status_counts = (
    fit_summary_df["status"].value_counts().to_dict()
    if len(fit_summary_df)
    else {}
)

manifest = {
    "pipeline": (
        "incremental-new-report-extraction-append-"
        "class-fit-checkpoint-inference"
    ),
    "new_input_path": str(NEW_INPUT_PATH),
    "old_unique_events_before_run": len(existing_unique_before_run),
    "new_source_rows": len(new_reports_source),
    "new_rows_appended_this_run": len(newly_appended_ids),
    "newly_appended_ids": newly_appended_ids,
    "combined_unique_events": len(combined_events),
    "raw_extraction_history": {
        "path": str(PREDICTED_EVENTS_PATH),
        "sha256_after_run": sha256_file(PREDICTED_EVENTS_PATH),
        "policy": (
            "append-only; old physical lines are never "
            "rewritten or reordered"
        ),
    },
    "extraction": {
        "model": QWEN_MODEL,
        "batch_size": EXTRACTION_BATCH_SIZE,
        "max_context_tokens": MAX_CONTEXT_TOKENS,
        "first_output_ceiling": FAST_MAX_NEW_TOKENS,
        "retry_output_ceiling": RETRY_MAX_NEW_TOKENS,
        "classification_prediction": False,
    },
    "class_fit": {
        "task": "candidate fit check only; no reclassification",
        "statuses": ["fit", "not_fit", "uncertain"],
        "counts": {str(k): int(v) for k, v in status_counts.items()},
    },
    "checkpoint": {
        "path": str(PHTKG_CHECKPOINT_PATH),
        "sha256": sha256_file(PHTKG_CHECKPOINT_PATH),
        "training_year_min": year_min,
        "training_year_max": year_max,
        "embedding_dim": predicted_model.dimension,
        "config": PHTKG_CONFIG,
    },
    "clustering": {
        "method": "coarse-to-fine full KMeans plus full silhouette",
        "n_init": KMEANS_N_INIT,
        "min_pattern_size": MIN_PATTERN_SIZE,
        "chosen_patterns": int(n_patterns),
    },
    "outputs": {
        "final_enriched_jsonl": str(FINAL_ENRICHED_JSONL),
        "verified_retraining_jsonl": str(VERIFIED_RETRAIN_PATH),
        "class_fit_audit": str(VERIFY_AUDIT_PATH),
        "incremental_temporal_drift": str(
            INFERENCE_DIR / "incremental_new_event_entity_drift.csv"
        ),
    },
}

MANIFEST_PATH = INFERENCE_DIR / "incremental_inference_manifest.json"

MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Manifest:", MANIFEST_PATH)


## Result

After one run you have:

### 1. Persistent raw extraction history

`MyDrive/ai_harm_ground_truth_outputs/predicted_events.jsonl`

This file is literally:

```text
all previous extracted events
new event 1
new event 2
new event 3
...
```

Only unseen `report_id`s are appended.

### 2. Final all-together enriched inference output

`MyDrive/ai_harm_ground_truth_outputs/incremental_inference/predicted_events_ALL_with_verified_class_and_phtkg.jsonl`

It contains **all previous + new events**, in the same old-then-new order, with:

- harm-category fit audit
- `verified_harm_category`
- PHTKG event embedding
- PHTKG event score
- learned temporal representation
- next-year recurrence
- incremental entity-state evolution
- similar events
- fast latent pattern
- pattern probability
- novelty
- temporal pattern evolution

### 3. Verified data for a future final retrain

`events_ALL_verified_for_retraining.jsonl`

Use this only if you decide to retrain the final PHTKG after the class-fit audit.
